In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report


c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
random_state = 42

In [3]:
df = pd.read_csv('../Milestone 3 EDA/credit_card_cleaned.csv')
df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


## Baseline

In [4]:
X = df.drop(columns='Class')
y = df['Class']

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2, 
                                                    stratify=y,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

### Default settings on RF

In [6]:
# Train using optimal parameters
model = RandomForestClassifier(
    random_state=random_state,
    n_jobs=-1,
)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [7]:
y_pred = model.predict(X_test)

In [8]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.94      0.82      0.87        98

    accuracy                           1.00     56962
   macro avg       0.97      0.91      0.94     56962
weighted avg       1.00      1.00      1.00     56962



In [9]:
baseline_model_report_df = classification_report(y_test, y_pred, output_dict=True)

### Class Weight Balanced

In [10]:
# Train using optimal parameters
model = RandomForestClassifier(
    class_weight='balanced',
    random_state=random_state,
    n_jobs=-1
)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [11]:
y_pred = model.predict(X_test)

In [12]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.96      0.74      0.84        98

    accuracy                           1.00     56962
   macro avg       0.98      0.87      0.92     56962
weighted avg       1.00      1.00      1.00     56962



In [13]:
baseline_balanced_model_report_df = classification_report(y_test, y_pred, output_dict=True)

## Undersamplgin

In [14]:
fraud = df[df['Class'] == 1]
non_fraud = df[df['Class'] == 0]
len(fraud), len(non_fraud)

(492, 284315)

In [15]:
# Undersample majority class to a 1:10 ratio
non_fraud_downsampled = non_fraud.sample(n=len(fraud) * 10, random_state=random_state)
non_fraud_downsampled

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
138028,82450.0,1.314539,0.590643,-0.666593,0.716564,0.301978,-1.125467,0.388881,-0.288390,-0.132137,...,-0.170307,-0.429655,-0.141341,-0.200195,0.639491,0.399476,-0.034321,0.031692,0.76,0
63099,50554.0,-0.798672,1.185093,0.904547,0.694584,0.219041,-0.319295,0.495236,0.139269,-0.760214,...,0.202287,0.578699,-0.092245,0.013723,-0.246466,-0.380057,-0.396030,-0.112901,4.18,0
73411,55125.0,-0.391128,-0.245540,1.122074,-1.308725,-0.639891,0.008678,-0.701304,-0.027315,-2.628854,...,-0.133485,0.117403,-0.191748,-0.488642,-0.309774,0.008100,0.163716,0.239582,15.00,0
164247,116572.0,-0.060302,1.065093,-0.987421,-0.029567,0.176376,-1.348539,0.775644,0.134843,-0.149734,...,0.355576,0.907570,-0.018454,-0.126269,-0.339923,-0.150285,-0.023634,0.042330,57.00,0
148999,90434.0,1.848433,0.373364,0.269272,3.866438,0.088062,0.970447,-0.721945,0.235983,0.683491,...,0.103563,0.620954,0.197077,0.692392,-0.206530,-0.021328,-0.019823,-0.042682,0.00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249845,154606.0,-1.235823,-0.571536,1.620984,-2.648022,0.431772,1.315577,-0.258094,0.687327,-1.039031,...,0.163828,0.220473,-0.023553,-1.685719,0.301464,-0.260192,0.268870,0.102644,88.30,0
131485,79621.0,1.227929,0.065334,0.043523,0.109012,-0.394905,-1.307819,0.349677,-0.310790,-0.030846,...,-0.424623,-1.375408,0.178609,0.395001,0.078121,0.619247,-0.109945,0.009712,44.94,0
96599,65844.0,-0.878702,0.077973,0.345555,-1.646600,-0.090887,-0.930793,-0.136898,0.407723,-1.456907,...,0.367227,0.880295,-0.150938,0.039828,-0.197562,-0.330629,0.274606,0.090629,15.00,0
164556,116806.0,-1.296659,0.282590,1.651779,-1.503733,-1.575132,-0.079848,-0.673041,0.332639,0.226605,...,-0.141780,0.269977,-0.043272,-0.036735,-0.731235,0.545507,-0.173457,0.059645,45.51,0


In [16]:
# Combine and shuffle
df_reduced = pd.concat([fraud, non_fraud_downsampled]).sample(frac=1).reset_index(drop=True)
df_reduced

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,159595.0,0.058198,-0.568091,0.572386,-2.026601,-1.712810,-0.404623,-0.854774,0.309016,-1.855603,...,0.308923,1.264259,0.186943,0.020606,-0.771501,-0.014161,0.054398,0.060696,76.50,0
1,170624.0,-6.729411,6.202271,-4.858542,-1.077134,-2.690236,-1.862082,-1.962344,3.176962,1.959044,...,0.269847,0.973180,0.576281,-0.033582,0.266224,-0.159472,0.893651,0.850891,2.23,0
2,137964.0,0.000369,0.410913,-1.752857,-1.679642,1.872037,-1.511905,2.368843,-0.858596,-0.386364,...,0.534175,1.839547,-0.145707,-0.282628,-0.977345,-0.123117,-0.024836,0.100298,94.00,0
3,26961.0,-23.237920,13.487386,-25.188773,6.261733,-17.345188,-4.534989,-17.100492,15.374630,-3.845567,...,1.769708,-1.691973,-1.045673,0.143386,1.611577,-0.221576,1.481233,0.438125,99.99,1
4,135095.0,0.232512,0.938944,-4.647780,3.079844,-1.902655,-1.041408,-1.020407,0.547069,-1.105990,...,0.911373,1.042929,0.999394,0.901260,-0.452093,0.192959,0.180859,-0.029315,345.00,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5407,75353.0,-0.347115,0.642984,1.437025,0.043874,0.016548,-0.333314,0.573444,0.031889,-0.359167,...,0.254176,0.717301,-0.161641,0.128705,0.057913,0.506090,-0.000468,0.016406,33.80,0
5408,67859.0,-0.381367,1.056180,1.307479,-0.051254,0.053216,-0.668698,0.578171,0.097346,-0.592646,...,-0.217722,-0.610316,-0.025837,0.267104,-0.204733,0.055972,0.239640,0.087102,1.99,0
5409,114782.0,-6.171083,3.664822,-4.206587,-0.590427,-1.965903,-1.708880,-1.165982,2.361165,1.870738,...,0.090239,0.827777,-0.068012,1.162462,-0.047745,-0.235597,0.923786,1.317223,1.00,0
5410,129764.0,-2.434004,3.225947,-6.596282,3.593161,-1.079452,-1.739741,-0.047420,0.301424,-1.779434,...,-0.035491,-0.419178,0.157436,-0.714849,0.468859,-0.348522,0.420036,-0.327643,362.55,1


In [17]:
X = df_reduced.drop(columns='Class')
y = df_reduced['Class']

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2, 
                                                    stratify=y,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

In [19]:
cv = StratifiedKFold(n_splits=5)

### Recall as metric

In [20]:
# Parameter tuning
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    
    model = RandomForestClassifier(
        class_weight='balanced',
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    return scores.mean()


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

top_5_recall_df = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_recall_df)

[I 2026-04-14 20:29:13,125] A new study created in memory with name: no-name-48d584a9-9ea2-48a3-bb33-e7bf5a264451
[I 2026-04-14 20:29:15,083] Trial 0 finished with value: 0.8376501135994807 and parameters: {'n_estimators': 67, 'max_depth': 12, 'min_samples_split': 6}. Best is trial 0 with value: 0.8376501135994807.
[I 2026-04-14 20:29:16,468] Trial 1 finished with value: 0.8579032781564427 and parameters: {'n_estimators': 94, 'max_depth': 3, 'min_samples_split': 8}. Best is trial 1 with value: 0.8579032781564427.
[I 2026-04-14 20:29:18,044] Trial 2 finished with value: 0.8325219084712755 and parameters: {'n_estimators': 185, 'max_depth': 2, 'min_samples_split': 8}. Best is trial 1 with value: 0.8579032781564427.
[I 2026-04-14 20:29:20,350] Trial 3 finished with value: 0.8452450503083415 and parameters: {'n_estimators': 87, 'max_depth': 17, 'min_samples_split': 10}. Best is trial 1 with value: 0.8579032781564427.
[I 2026-04-14 20:29:20,554] Trial 4 finished with value: 0.837585199610516

,number,value,datetime_start,datetime_complete,duration,params_max_depth,params_min_samples_split,params_n_estimators,state
60,60,0.860435,2026-04-14 20:29:46.842733,2026-04-14 20:29:47.092685,0 days 00:00:00.249952,3,9,51,COMPLETE
1,1,0.857903,2026-04-14 20:29:15.083652,2026-04-14 20:29:16.468687,0 days 00:00:01.385035,3,8,94,COMPLETE
32,32,0.857903,2026-04-14 20:29:36.563653,2026-04-14 20:29:36.825955,0 days 00:00:00.262302,3,8,61,COMPLETE
26,26,0.857903,2026-04-14 20:29:34.623611,2026-04-14 20:29:34.919969,0 days 00:00:00.296358,3,9,61,COMPLETE
74,74,0.857903,2026-04-14 20:29:53.255781,2026-04-14 20:29:53.583631,0 days 00:00:00.327850,3,8,79,COMPLETE


In [21]:
# Train using optimal parameters
model_1 = RandomForestClassifier(
    class_weight='balanced',
    n_estimators=top_5_recall_df.head(1)['params_n_estimators'].item(),
    max_depth=top_5_recall_df.head(1)['params_max_depth'].item(),
    min_samples_split=top_5_recall_df.head(1)['params_min_samples_split'].item(),
    random_state=random_state
)
model_1.fit(X_train, y_train)

,n_estimators,51
,criterion,'gini'
,max_depth,3
,min_samples_split,9
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [22]:
y_pred = model_1.predict(X_test)

In [23]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       985
           1       0.94      0.91      0.92        98

    accuracy                           0.99      1083
   macro avg       0.96      0.95      0.96      1083
weighted avg       0.99      0.99      0.99      1083



In [24]:
recall_model_report_df = classification_report(y_test, y_pred, output_dict=True)

## F1 as metric

In [25]:
# Parameter tuning
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    
    model = RandomForestClassifier(
        class_weight='balanced',
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    return scores.mean()


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

top_5_f1_df = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_f1_df)

[I 2026-04-14 20:30:03,645] A new study created in memory with name: no-name-84236084-9cd3-4b5e-b2a6-b972d61141bb
[I 2026-04-14 20:30:04,094] Trial 0 finished with value: 0.9021505069256033 and parameters: {'n_estimators': 55, 'max_depth': 7, 'min_samples_split': 10}. Best is trial 0 with value: 0.9021505069256033.
[I 2026-04-14 20:30:05,954] Trial 1 finished with value: 0.9043444307494408 and parameters: {'n_estimators': 197, 'max_depth': 20, 'min_samples_split': 8}. Best is trial 1 with value: 0.9043444307494408.
[I 2026-04-14 20:30:06,342] Trial 2 finished with value: 0.8923152190852353 and parameters: {'n_estimators': 117, 'max_depth': 2, 'min_samples_split': 5}. Best is trial 1 with value: 0.9043444307494408.
[I 2026-04-14 20:30:08,216] Trial 3 finished with value: 0.9055778789627549 and parameters: {'n_estimators': 195, 'max_depth': 15, 'min_samples_split': 7}. Best is trial 3 with value: 0.9055778789627549.
[I 2026-04-14 20:30:09,565] Trial 4 finished with value: 0.9043444307494

,number,value,datetime_start,datetime_complete,duration,params_max_depth,params_min_samples_split,params_n_estimators,state
82,82,0.908377,2026-04-14 20:31:08.318077,2026-04-14 20:31:08.707907,0 days 00:00:00.389830,7,3,44,COMPLETE
79,79,0.908377,2026-04-14 20:31:07.241587,2026-04-14 20:31:07.615085,0 days 00:00:00.373498,7,3,44,COMPLETE
92,92,0.908217,2026-04-14 20:31:12.664470,2026-04-14 20:31:13.071397,0 days 00:00:00.406927,7,3,54,COMPLETE
93,93,0.908217,2026-04-14 20:31:13.071397,2026-04-14 20:31:13.526399,0 days 00:00:00.455002,7,3,57,COMPLETE
58,58,0.907356,2026-04-14 20:30:55.663834,2026-04-14 20:30:56.417245,0 days 00:00:00.753411,11,10,78,COMPLETE


In [26]:
# Train using optimal parameters
model_2 = RandomForestClassifier(
    class_weight='balanced',
    n_estimators=top_5_f1_df.head(1)['params_n_estimators'].item(),
    max_depth=top_5_f1_df.head(1)['params_max_depth'].item(),
    min_samples_split=top_5_f1_df.head(1)['params_min_samples_split'].item(),
    random_state=random_state
)
model_2.fit(X_train, y_train)

,n_estimators,44
,criterion,'gini'
,max_depth,7
,min_samples_split,3
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [27]:
y_pred = model_2.predict(X_test)

In [28]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       985
           1       0.98      0.90      0.94        98

    accuracy                           0.99      1083
   macro avg       0.98      0.95      0.97      1083
weighted avg       0.99      0.99      0.99      1083



In [29]:
f1_model_report_df = classification_report(y_test, y_pred, output_dict=True)

### Analyze reports

In [38]:
# 2. Convert and transpose
recall_report_df = pd.DataFrame(recall_model_report_df).transpose()
f1_report_df = pd.DataFrame(f1_model_report_df).transpose()

# 3. Combine multiple reports (e.g., adding a 'Model' column for identification)
combined_df = pd.concat([recall_report_df, f1_report_df], keys=['CC RF optimized for Recall', 'CC RF optimized for F1'])

In [39]:
combined_df

precision    recall  f1-score  \
CC RF optimized for Recall 0              0.990891  0.993909  0.992397   
                           1              0.936842  0.908163  0.922280   
                           accuracy       0.986150  0.986150  0.986150   
                           macro avg      0.963866  0.951036  0.957339   
                           weighted avg   0.986000  0.986150  0.986052   
CC RF optimized for F1     0              0.989930  0.997970  0.993933   
                           1              0.977778  0.897959  0.936170   
                           accuracy       0.988920  0.988920  0.988920   
                           macro avg      0.983854  0.947964  0.965052   
                           weighted avg   0.988830  0.988920  0.988706   

                                            support  
CC RF optimized for Recall 0              985.00000  
                           1               98.00000  
                           accuracy         0.98615  
                           macro avg     1083.00000  
                           weighted avg  1083.00000  
CC RF optimized for F1     0              985.00000  
                           1               98.00000  
                           accuracy         0.98892  
                           macro avg     1083.00000  
                           weighted avg  1083.00000

In [32]:
# Recall model
importances = model_1.feature_importances_
feature_names = df.drop(columns='Class').columns
feature_imp_recall_df = pd.DataFrame({'Feature': feature_names, 'Gini Importance': importances}).sort_values(
    'Gini Importance', ascending=False)
feature_imp_recall_df

,Feature,Gini Importance
14,V14,0.308357
10,V10,0.136394
17,V17,0.099583
4,V4,0.095286
3,V3,0.077117
16,V16,0.076696
12,V12,0.075937
2,V2,0.022241
9,V9,0.019767
7,V7,0.015803


In [33]:
# Recall model
importances = model_2.feature_importances_
feature_names = df.drop(columns='Class').columns
feature_imp_f1_df = pd.DataFrame({'Feature': feature_names, 'Gini Importance': importances}).sort_values(
    'Gini Importance', ascending=False)
feature_imp_f1_df

,Feature,Gini Importance
14,V14,0.262059
4,V4,0.106912
10,V10,0.093380
17,V17,0.091772
12,V12,0.086796
16,V16,0.076890
3,V3,0.075675
2,V2,0.025106
7,V7,0.022226
9,V9,0.020086
